# Publication figures · Bogotá stochastic accessibility

Run all cells in Colab. Exports the eight numbered main-text figures as `PNG`, `PDF`, and `EPS` into one versioned ZIP on Drive. Figures 1–6 are converted from their **existing archived PNG** images; the resulting PDF/EPS contain raster images and do not gain vector resolution. Figures 7–8 are redrawn as vector PDF/EPS from their recorded plot data; Figure 8 requires the **exact** Gate 24 intervals CSV in Drive. No analysis or numerical estimand is recalculated here.

If `/content/colombia-stochastic-access` is missing, upload one repository ZIP downloaded from your private GitHub repository. The notebook never asks for or stores a GitHub token. It does not modify the manuscript.

In [ ]:
%pip -q install numpy pandas matplotlib pillow
from pathlib import Path
import os, shutil, zipfile, runpy
import pandas as pd
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from google.colab import drive, files
drive.mount("/content/drive", force_remount=False)
ROOT = Path(os.environ.get("STOCHASTIC_PROJECT_ROOT", "/content/colombia-stochastic-access"))
if not (ROOT / "paper/figures").is_dir():
    print("Select one ZIP of the private repository (GitHub > Code > Download ZIP).")
    uploaded = files.upload()
    if len(uploaded) != 1: raise ValueError("Upload exactly one ZIP.")
    name, content = next(iter(uploaded.items()))
    zippath = Path("/content/project-source.zip"); zippath.write_bytes(content); del content
    with zipfile.ZipFile(zippath) as archive:
        members = archive.namelist()
        roots = {x.split("/")[0] for x in members if x and not x.startswith("__MACOSX")}
        if len(roots) != 1: raise ValueError("Expected a single repository folder in ZIP.")
        target = Path("/content").resolve()
        for m in members:
            dst = (target / m).resolve()
            if dst != target and target not in dst.parents: raise ValueError("Unsafe path in ZIP.")
        archive.extractall(target)
    ROOT = Path("/content") / roots.pop()
PAPER = ROOT / "paper"
DRIVE = Path("/content/drive/MyDrive/colombia-stochastic-access-data/bogota_em2023")
GATE24 = Path(os.environ.get("GATE24_INTERVALS", DRIVE / "results/gate24/boundary_mechanism_intervals.csv"))
if not GATE24.exists(): raise FileNotFoundError(f"Gate 24 exact intervals missing: {GATE24}")
required = {"purpose_group","term","point_estimate","simultaneous_ci_lower","simultaneous_ci_upper"}
intervals = pd.read_csv(GATE24)
if len(intervals) != 12 or not required.issubset(intervals.columns):
    raise ValueError("Gate 24 exact CSV has unexpected columns or row count")
OUT = Path(os.environ.get("FIGURE_EXPORT_DIRECTORY", DRIVE / "results/publication_figures_gate24"))
OUT.mkdir(parents=True, exist_ok=True)
print("Source:", PAPER, "\\nExact Gate 24 CSV:", GATE24, "\\nExports:", OUT)

In [ ]:
# Raster originals: PDF/EPS encapsulate original pixels at final printed dimensions.
# No upscaling or invented vector paths.
rasters = [
    "figure_study_area_multiscale_scopes",
    "figure_1_scope_sex_gaps",
    "figure_2_paired_scope_amplification",
    "figure_boundary_response_total_gap",
    "figure_boundary_response_decomposition",
    "figure_3_external_municipality_kernel_contributions",
]
for stem in rasters:
    src = PAPER / "figures" / (stem + ".png")
    if not src.exists(): raise FileNotFoundError(src)
    with Image.open(src) as im:
        im = im.convert("RGB")
        w,h = im.size
        shutil.copy2(src, OUT / (stem+".png"))
        fig = plt.figure(figsize=(w/300,h/300),dpi=300)
        ax = fig.add_axes([0,0,1,1]); ax.imshow(im, interpolation="none"); ax.axis("off")
        fig.savefig(OUT/(stem+".pdf"),facecolor="white")
        fig.savefig(OUT/(stem+".eps"),facecolor="white")
        plt.close(fig)
    print(stem, f"{w}x{h} source pixels; PDF/EPS raster-embedded")
# Vector source charts are redrawn from their saved numerical values.
runpy.run_path(str(PAPER / "scripts/plot_gate23_validation.py"),run_name="__main__")
for ext in ("png","pdf","eps"):
    shutil.copy2(PAPER/"figures"/("figure_7_validation_diagnostics."+ext),OUT/("figure_7_validation_diagnostics."+ext))
os.environ["GATE24_INTERVALS"] = str(GATE24)
os.environ["FIGURE_OUTPUT"] = str(OUT)
runpy.run_path(str(PAPER / "scripts/plot_gate24_boundary_mechanism.py"),run_name="__main__")
print("All eight publication figures exported in three formats.")

In [ ]:
from datetime import datetime, timezone
import json, hashlib
figures = rasters + ["figure_7_validation_diagnostics","figure_boundary_mechanism"]
records = []
for stem in figures:
    item = {"figure":stem,"source_type":"vector-generated" if stem not in rasters else "raster-embedded"}
    for ext in ("png","pdf","eps"):
        p = OUT/(stem+"."+ext)
        if not p.is_file() or p.stat().st_size < 1000: raise RuntimeError("Missing/empty output: "+str(p))
        item[ext+"_bytes"] = p.stat().st_size
        item[ext+"_sha256"] = hashlib.sha256(p.read_bytes()).hexdigest()
    records.append(item)
manifest = {"date_utc":datetime.now(timezone.utc).isoformat(),"gate24_csv":str(GATE24),
            "gate24_csv_sha256":hashlib.sha256(GATE24.read_bytes()).hexdigest(),
            "figure_count":len(figures),"files":records,
            "note":"Figures 1–6 PDF/EPS embed their original PNG raster; 7–8 are vector-generated."}
(OUT/"figure_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
archive = OUT/"bogota_figures_png_pdf_eps.zip"
with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for stem in figures:
        for ext in ("png","pdf","eps"): z.write(OUT/(stem+"."+ext),arcname=stem+"."+ext)
    z.write(OUT/"figure_manifest.json",arcname="figure_manifest.json")
assert zipfile.ZipFile(archive).testzip() is None
print("ZIP:",archive, "size MB:",round(archive.stat().st_size/1e6,2))
print(pd.DataFrame(records)[["figure","source_type","png_bytes","pdf_bytes","eps_bytes"]].to_string(index=False))